In [1]:
from clickhouse_driver import Client
import pandas as pd

In [ ]:
client = Client(
    host='',
    user='default',
    password='',
    secure=True
)

In [8]:
client.execute("""
CREATE TABLE IF NOT EXISTS helloworld.realtime_taxi_trips
(
    trip_id UInt64,
    pickup_time DateTime,
    pickup_location String,
    dropoff_location String,
    passenger_count UInt8,
    trip_distance Float64,
    fare_amount Float64,
    payment_type String,
    created_at DateTime DEFAULT now()
)
ENGINE = MergeTree
ORDER BY pickup_time
""")

[]

In [5]:
import random
import time
from datetime import datetime


locations = [
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Jersey City"
]

payments = [
    "CARD",
    "CASH",
    "WALLET"
]


def generate_trip():
    return {
        "trip_id": random.randint(1000000,9999999),
        "pickup_time": datetime.now(),
        "pickup_location": random.choice(locations),
        "dropoff_location": random.choice(locations),
        "passenger_count": random.randint(1,5),
        "trip_distance": round(random.uniform(0.5,20),2),
        "fare_amount": round(random.uniform(5,80),2),
        "payment_type": random.choice(payments)
    }

In [9]:
while True:

    batch = []

    for i in range(100):
        batch.append(generate_trip())

    df = pd.DataFrame(batch)

    client.execute(
        """
        INSERT INTO helloworld.realtime_taxi_trips
        (trip_id,
         pickup_time,
         pickup_location,
         dropoff_location,
         passenger_count,
         trip_distance,
         fare_amount,
         payment_type)
        VALUES
        """,
        df.to_dict("records")
    )

    print(
        f"Inserted {len(df)} trips at {datetime.now()}"
    )

    time.sleep(5)

Inserted 100 trips at 2026-06-12 11:46:23.215660
Inserted 100 trips at 2026-06-12 11:46:29.399932
Inserted 100 trips at 2026-06-12 11:46:35.639630
Inserted 100 trips at 2026-06-12 11:46:41.845906
Inserted 100 trips at 2026-06-12 11:46:48.071345
Inserted 100 trips at 2026-06-12 11:46:54.206532
Inserted 100 trips at 2026-06-12 11:47:00.340225
Inserted 100 trips at 2026-06-12 11:47:06.563442


KeyboardInterrupt: 